# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 17 — The attribution analysis on partitions drawn without the defective splitter

The manuscript shows that ordering scaffold groups by descending size and filling greedily
produces test sets that overlap at a mean Jaccard of $0.634$, so that fifteen partitions carry
the weight of $1.26$ independent observations, and it reports a variant that orders the groups
at random as the remedy. Every attribution result in the paper nonetheless comes from a single
partition drawn with the rule the paper discredits.

That is not an omission to be declared. It is the paper failing to apply its own criticism to
its own main analysis, and a reader will ask why the corrected splitter, which is already in
the manuscript, was not used.

This notebook redraws three partitions with the random-order variant and repeats the
measurements the headline rests on. It also settles two checks the manuscript currently
describes without running.

**Contents**

1. Environment and shared definitions
2. Three partitions from the corrected splitter, and their overlap
3. Models
4. The headline comparisons, per partition
5. The regime decomposition, per partition
6. Difference of differences for the specificity check
7. A matched arm for the permutation control
8. Summary

*Approximate runtime: two to three hours on a GPU. Fifteen models are trained; every stage
checkpoints.*

## Pre-registered readings

Fixed before the notebook is run. Every reading has three branches, because with five models
per partition and eight in the matched arm the most likely single outcome is an interval that
contains both zero and the original point estimate. A two-branch reading would have to be
resolved after the numbers are seen, which is the failure this table exists to prevent.

### Composition first, and one exclusion to declare

The Murcko scaffold of an acyclic molecule is the empty string, so all acyclic compounds form
a single group, and where that group lands decides what an evaluation set is made of. Both
splitters place it in training. That keeps the comparison to one changed variable, and it
means no evaluation set in this work contains an acyclic compound: the share of alert atoms
carried by the fused-aromatic pattern is measured on a subset with no purely aliphatic
mutagens in it. Section 4 computes the same share over all mutagenic compounds so that the
difference can be reported as a limitation.

Before any reading below is applied, each partition is checked for comparability on the
number of evaluation molecules, the chance level, the share of alert atoms carried by the
fused-aromatic pattern, and the share of acyclic compounds. **If a partition differs markedly
from the others on these, its comparison is reported as not comparable rather than as a
replication failure.**

### The headline, AUROC difference of GIN over degree

| result | reading |
|---|---|
| positive and excluding zero on all three | the headline replicates; the single-partition limitation is discharged |
| positive on all three, at least one interval containing zero | the direction replicates and the size is not resolved at five models per partition; reported as such, with the subsampled reference below |
| any partition negative and excluding zero | the headline does not survive the corrected splitter; the abstract changes and the original result is reported as partition-dependent |

### Tie fraction for the degree ranking

| result | reading |
|---|---|
| near $0.87$ on every partition | it follows from how few values degree takes, as claimed |
| varying by more than $0.10$ across partitions | the measure's behaviour is partition-dependent and that goes in the results |

### Regime decomposition

| result | reading |
|---|---|
| both signs hold on all three | the decomposition replicates and the two-effect claim stands |
| signs hold, some intervals contain zero | direction replicates, size not resolved; reported with the range |
| either sign flips on a comparable partition | the claim is narrowed to the original partition and reported as such |

### Difference of differences, specificity

The statistic is $(\text{non-mutagenic} - \text{degree}) - (\text{mutagenic} - \text{degree})$,
so the pre-registered direction is **negative**.

| result | reading |
|---|---|
| negative and excluding zero | models rank alert atoms lower where they do not carry the endpoint; the hypothesis is supported |
| positive and excluding zero | the opposite of the pre-registered direction, reported as such |
| containing zero | the check is underpowered or there is no effect; reported as not resolved, never as evidence of absence |

### Matched permutation arm

The comparison is between this arm and the permuted arm, both fitted in the same regime.
The untrained value is not an anchor: that network was never fitted at all, and the question
is whether the drop follows from the labels or from how the fitting was done. The interval is
a $t$ interval, since a percentile bootstrap of a mean is anti-conservative at $n = 8$.

| result | reading |
|---|---|
| interval lies entirely above the permuted arm | real labels in the same regime give higher agreement, so the drop belongs to the labels and the interpretive paragraph stands |
| interval contains the permuted arm | the drop is attributable to the training regime and the paragraph is withdrawn |
| interval lies entirely below the permuted arm | unexpected under either account; reported as observed, and the paragraph is withdrawn pending an explanation |

### On the standing of this table

The table is committed before the run. The manuscript states that its other pre-specification
claims carry no timestamped record; for these analyses this cell is that record, and the
commit hash should be quoted in the manuscript. A pre-registration living only inside the
notebook that produces the numbers is an assertion, not a record.

## 1. Environment and shared definitions

In [1]:
!pip install -q rdkit
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 62.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = "/content/drive/MyDrive/ames_pipeline"
DATADIR, R, EXPL, MODELS = f"{WORK}/data", f"{WORK}/results", f"{WORK}/explain", f"{WORK}/models"
CKPT = f"{WORK}/ckpt_part17"
for d in (R, EXPL, MODELS, CKPT): os.makedirs(d, exist_ok=True)
print("checkpoints:", CKPT)

Mounted at /content/drive
checkpoints: /content/drive/MyDrive/ames_pipeline/ckpt_part17


In [3]:
%%writefile /content/ames9.py
"""Definitions shared across Part 9, matching the protocol used in Parts 2 to 5."""
import random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator as rfg
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*"); warnings.filterwarnings("ignore")
FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']
morgan_gen = rfg.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return ""
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try: return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception: return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs = list(groups.values()); rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te]

def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr

def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None: X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)

def one_hot(x, allowable):
    if x not in allowable: x = allowable[-1]
    return [int(x == a) for a in allowable]

def atom_features(a):
    f  = one_hot(a.GetSymbol(), ATOM_LIST) + one_hot(str(a.GetHybridization()), HYBRIDIZATION)
    f += [a.GetDegree(), a.GetFormalCharge(), int(a.GetIsAromatic()),
          a.GetTotalNumHs(), int(a.IsInRing())]
    return f

def bond_features(b):
    bt = b.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(b.GetIsConjugated()), int(b.IsInRing())]

def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_features(b)
        ei += [[i, j], [j, i]]; ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)

def build_dataset(d):
    g = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [x for x in g if x is not None]

class GNN(nn.Module):
    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn": conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else: raise ValueError(kind)
            self.convs.append(conv); self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, data):
        x, ei, ea, b = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(x, ei, ea) if self.use_edge_attr else conv(x, ei)
            h = F.relu(bn(h)); h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, b), global_add_pool(x, b)], dim=1)
        return self.head(g).squeeze(-1)

def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4, use_edge_attr=True):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads, use_edge_attr=use_edge_attr).to(DEVICE)

class TensorView:
    def __init__(self, x, edge_index, edge_attr, batch):
        self.x, self.edge_index, self.edge_attr, self.batch = x, edge_index, edge_attr, batch

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    for d in loader:
        d = d.to(DEVICE)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(model(d)).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def train_gnn(kind, params, seed, tr_set, va_set, pos_weight,
              max_epochs=250, patience=30, use_edge_attr=True, epoch_hook=None):
    """The protocol used for every model in this study."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    opt = torch.optim.Adam(model.parameters(), lr=params["lr"],
                           weight_decay=params["weight_decay"])
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    tl = DataLoader(tr_set, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    vl = DataLoader(va_set, batch_size=256)
    best, best_state, waited = -1.0, None, 0
    for ep in range(max_epochs):
        model.train()
        for d in tl:
            d = d.to(DEVICE); opt.zero_grad()
            crit(model(d), d.y.view(-1)).backward(); opt.step()
        y, p = evaluate(model, vl)
        auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else 0.5
        if epoch_hook is not None: epoch_hook(ep, model, auc)
        if auc > best:
            best, waited = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience: break
    model.load_state_dict(best_state); model.eval()
    return model, best

def train_to_fit(kind, params, seed, dataset, n_epochs, dropout=0.0, weight_decay=0.0,
                 lr=None):
    """Fit a target with regularisation and early stopping switched off."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], dropout,
                      params.get("residual", True))
    opt = torch.optim.Adam(model.parameters(), lr=lr or params["lr"],
                           weight_decay=weight_decay)
    crit = nn.BCEWithLogitsLoss()
    ld = DataLoader(dataset, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    last = float("nan")
    for ep in range(n_epochs):
        model.train(); tot = 0.0
        for d in ld:
            d = d.to(DEVICE); opt.zero_grad()
            loss = crit(model(d), d.y.view(-1)); loss.backward(); opt.step()
            tot += loss.item()
        last = tot / max(len(ld), 1)
    model.eval(); return model, last

Writing /content/ames9.py


In [4]:
import sys, json, glob, time, itertools
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, torch
from ames9 import *
from scipy import stats
from sklearn.metrics import roc_auc_score as _auc

SEED = 42
N_MODELS_PER_PARTITION = 5      # enough to average initialisation noise, not to profile it
K_FRAC, N_TIEBREAK, N_BOOT, N_IG_STEPS = 0.30, 200, 10000, 32
HP = json.load(open(f"{R}/hpo_gnn.json"))["gin"]

def cached(name, fn):
    p = f"{R}/{name}"
    if os.path.exists(p):
        print(f"  {name}: loaded from disk"); return pd.read_csv(p)
    out = fn()
    if "note" in out.columns or not len(out):
        print(f"  {name}: stage could not run, not cached"); return out
    out.to_csv(p, index=False); print(f"  {name}: written"); return out

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in ["train.csv","valid.csv","test.csv"]],
               ignore_index=True)[["Drug_ID","smiles","label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)
assert len(df) == 7255, f"expected 7255 compounds after curation, got {len(df)}"
print(f"{len(df)} compounds")

# the configuration was selected on the designated partition, which was drawn with the
# size-ordered splitter. It is reused here unchanged so that only the splitter varies;
# if test performance falls on the new partitions this is a candidate explanation and
# has to be separated from partition difficulty.
print("hyperparameters: selected once on the designated partition, reused unchanged")

7255 compounds
hyperparameters: selected once on the designated partition, reused unchanged


## 2. Three partitions from the corrected splitter, and their overlap

The splitter used throughout the manuscript orders scaffold groups by descending size, so the
same large groups reach the training set every time and the test set is drawn repeatedly from
the pool of small scaffolds. The variant below orders them at random, which is the remedy the
manuscript reports. The Jaccard indices between the three test sets are computed here so that
the claim of independence is measured rather than assumed.

In [5]:
# Murcko scaffolds are needed by the splitter, the disjointness checks, the composition
# table and the alert-share cell. Computing them once also removes the risk of two call
# sites disagreeing.
SCAF = {s: safe_scaffold(s) for s in df.canonical}
print(f"scaffolds computed for {len(SCAF)} compounds")

def scaffold_split_random_order(d, seed, frac=(0.8, 0.1, 0.1)):
    """Scaffold-disjoint, with groups shuffled rather than ordered by size.

    The intended difference from make_scaffold_split is the sort key. That one uses -len(g),
    so every large group reaches the training set of every partition and the test set is
    filled from the same small ones. Everything else is held fixed, because the manuscript's
    claim is that the defect belongs to the ordering rule and not to scaffold partitioning
    itself; changing anything else would confound the comparison.

    Two groups cannot be shuffled without breaking the split, and both go to training here
    for the same reason. The Murcko scaffold of an acyclic molecule is the empty string, so
    all acyclic compounds form one group of 1,260. The largest cyclic scaffold group holds
    1,135. Each is larger than the whole validation or test budget of about 725, so placing
    either outside training does not produce a 10% test set: it produces a 16% one made of a
    single scaffold. The size-ordered rule sends both to training by construction, being
    largest first. Pinning any group that exceeds the smaller of the two budgets reproduces
    that behaviour without importing the ordering, and leaves the shuffle to decide
    everything the shuffle can decide."""
    from collections import defaultdict
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[SCAF[smi]].append(i)
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    budget = min(b - a, n - b)              # the smaller of the validation and test targets

    oversized, rest = [], []
    for k_, g in groups.items():
        (oversized if len(g) > budget else rest).append((k_, g))
    rng = np.random.default_rng(seed); rng.shuffle(rest)

    tr = [i for _, g in oversized for i in g]
    va, te = [], []
    for _, g in rest:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te], [(k_ or "(acyclic)", len(g)) for k_, g in oversized]

def composition(frame, name):
    """What an evaluation set is made of, so that a sign change can be diagnosed."""
    n_acyc = sum(1 for s in frame.canonical if SCAF[s] == "")
    return {"set": name, "n": len(frame), "acyclic": n_acyc,
            "acyclic_frac": n_acyc/len(frame), "mutagenic_frac": float(frame.label.mean())}

from collections import Counter
_cyc = Counter(v for v in SCAF.values() if v != "")
big_scaf, big_n = _cyc.most_common(1)[0]

PART_SEEDS = [201, 202, 203]
PARTS, comp = {}, []
n_acyclic_total = sum(1 for s in df.canonical if SCAF[s] == "")
print(f"acyclic compounds (empty Murcko scaffold): {n_acyclic_total} "
      f"({100*n_acyclic_total/len(df):.1f}% of the dataset), assigned to training throughout")
print()
for sd in PART_SEEDS:
    tr, va, te, pinned = scaffold_split_random_order(df, sd)
    tr, va, te = [x.reset_index(drop=True) for x in (tr, va, te)]
    PARTS[sd] = (tr, va, te)
    if sd == PART_SEEDS[0]:
        print("groups pinned to training because they exceed the validation and test budget:")
        for nm, sz in sorted(pinned, key=lambda x: -x[1]):
            print(f"    {sz:5d} compounds  {nm[:56]}")
        print()
    for nm, part in [("train", tr), ("valid", va), ("test", te)]:
        comp.append({"partition": sd, **composition(part, nm)})
    # the split must remain scaffold-disjoint in every direction
    S = {nm: {SCAF[s] for s in p.canonical} for nm, p in
         [("train", tr), ("valid", va), ("test", te)]}
    for x, y in [("train","valid"), ("train","test"), ("valid","test")]:
        assert not (S[x] & S[y]), f"seed {sd}: scaffolds shared between {x} and {y}"
    # and the proportions must not have collapsed
    f_te = len(te)/len(df); f_va = len(va)/len(df)
    assert 0.07 <= f_te <= 0.14, f"seed {sd}: test fraction {f_te:.3f} outside 0.07-0.14"
    assert 0.06 <= f_va <= 0.15, f"seed {sd}: valid fraction {f_va:.3f} outside 0.06-0.15"
    # the acyclic group is settled, but the largest cyclic group can still shift a split
    where = next(nm for nm, p in [("train", tr), ("valid", va), ("test", te)]
                 if any(SCAF[s] == big_scaf for s in p.canonical))
    print(f"  seed {sd}: largest cyclic scaffold group has {big_n} compounds, in {where}")
    print(f"  seed {sd}: train {len(tr)} valid {len(va)} test {len(te)} "
          f"({f_te:.3f}) | test mutagenic {te.label.mean():.3f}")

comp = pd.DataFrame(comp)
comp.to_csv(f"{R}/table85b_partition_composition.csv", index=False)
print()
print(comp[comp.set == "test"].round(4).to_string(index=False))

rows = []
for a, b in itertools.combinations(PART_SEEDS, 2):
    A = set(PARTS[a][2].canonical); B = set(PARTS[b][2].canonical)
    rows.append({"pair": f"{a}-{b}", "jaccard": len(A & B)/len(A | B)})
prim = pd.read_csv(f"{R}/primary_test.csv")
for sd in PART_SEEDS:
    A = set(PARTS[sd][2].canonical); B = set(prim.canonical)
    rows.append({"pair": f"{sd}-primary", "jaccard": len(A & B)/len(A | B)})
ov = pd.DataFrame(rows)
ov.to_csv(f"{R}/table86_partition_overlap.csv", index=False)
print()
print(ov.round(4).to_string(index=False))
print()
print("  The manuscript reports 0.634 for the size-ordered splitter and 0.052 for the")
print("  random-order variant. If these three sit near 0.634 the implementation here does")
print("  not match the one that produced that figure and the comparison is void.")
between = ov[~ov.pair.str.contains("primary")]
assert len(between) == 3 and between.jaccard.max() < 0.30, \
    "test sets overlap like the size-ordered splitter; check the implementation"

scaffolds computed for 7255 compounds
acyclic compounds (empty Murcko scaffold): 1260 (17.4% of the dataset), assigned to training throughout

groups pinned to training because they exceed the validation and test budget:
     1260 compounds  (acyclic)
     1135 compounds  c1ccccc1

  seed 201: largest cyclic scaffold group has 1135 compounds, in train
  seed 201: train 5804 valid 725 test 726 (0.100) | test mutagenic 0.590
  seed 202: largest cyclic scaffold group has 1135 compounds, in train
  seed 202: train 5804 valid 725 test 726 (0.100) | test mutagenic 0.607
  seed 203: largest cyclic scaffold group has 1135 compounds, in train
  seed 203: train 5804 valid 725 test 726 (0.100) | test mutagenic 0.517

 partition  set   n  acyclic  acyclic_frac  mutagenic_frac
       201 test 726        0           0.0          0.5895
       202 test 726        0           0.0          0.6074
       203 test 726        0           0.0          0.5165

       pair  jaccard
    201-202   0.0606
    2

## 3. Models

Five GIN models per partition, with the configuration selected for the manuscript. Five is
enough to average initialisation noise; profiling the spread is not the question here, and
the twenty-model spread on the original partition is already reported.

In [6]:
MDL_PATH = f"{CKPT}/models.csv"
meta = pd.read_csv(MDL_PATH).to_dict("records") if os.path.exists(MDL_PATH) else []
done = {(int(r["partition"]), int(r["seed"])) for r in meta}
t0 = time.time()
for sd in PART_SEEDS:
    tr, va, te = PARTS[sd]
    tr_set, va_set = build_dataset(tr), build_dataset(va)
    te_loader = DataLoader(build_dataset(te), batch_size=256)
    y_tr = tr["label"].values
    pos_w = torch.tensor([(y_tr == 0).sum()/max((y_tr == 1).sum(), 1)],
                         dtype=torch.float, device=DEVICE)
    for i in range(N_MODELS_PER_PARTITION):
        msd = 41000 + sd*10 + i
        if (sd, msd) in done: continue
        m, va_auc = train_gnn("gin", HP, msd, tr_set, va_set, pos_w)
        y, p = evaluate(m, te_loader)
        torch.save({"state_dict": m.state_dict(), "valid_auc": float(va_auc)},
                   f"{MODELS}/gin_part{sd}_seed{msd}.pt")
        meta.append({"partition": sd, "seed": msd, "valid_auc": float(va_auc),
                     "test_roc_auc": _auc(y, p)})
        pd.DataFrame(meta).to_csv(MDL_PATH, index=False)
        del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f"  partition {sd} model {i+1}/{N_MODELS_PER_PARTITION}: "
              f"test {meta[-1]['test_roc_auc']:.4f}  ({(time.time()-t0)/60:.0f} min)")
meta = pd.DataFrame(meta)
print()
print(meta.groupby("partition").test_roc_auc.agg(["mean","std","count"]).round(4).to_string())
print()
print("  For comparison, the designated partition of the manuscript gives 0.785 and the")
print("  out-of-fold cross-validation 0.845. A corrected partition should sit nearer the")
print("  second if the gap was a property of the splitter.")


             mean     std  count
partition                       
201        0.8191  0.0069      5
202        0.8821  0.0101      5
203        0.8683  0.0081      5

  For comparison, the designated partition of the manuscript gives 0.785 and the
  out-of-fold cross-validation 0.845. A corrected partition should sit nearer the
  second if the gap was a property of the splitter.


## 4. The headline comparisons, per partition

Each partition defines its own evaluation set, its own chance level and its own baseline. The
quantities repeated here are the ones the abstract rests on: the agreement of the degree
ranking, the tie fraction it carries, and the difference between a trained model and it on
the threshold-free measure.

In [7]:
TOX = {"aromatic_nitro":"[a][N+](=O)[O-]","aromatic_amine":"[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
 "aromatic_nitroso":"[a][NX2]=[OX1]","alkyl_nitrite":"[CX4][OX2][NX2]=[OX1]",
 "nitrosamine":"[NX3][NX2]=[OX1]","epoxide":"C1OC1","aziridine":"C1NC1",
 "azide":"[NX2]=[NX2+]=[NX1-]","azo_aromatic":"[a][NX2]=[NX2][a]",
 "aliphatic_halide":"[CX4][F,Cl,Br,I]","hetero_hetero_bond":"[N,O;H1,H2][N,O;H1,H2]",
 "polycyclic_aromatic":"c1ccc2ccccc2c1","aromatic_hydroxylamine":"[a][NX3;H1][OX2;H1]",
 "michael_acceptor":"[CX3]=[CX3][CX3]=[OX1]","aromatic_azoxy":"[a][NX2]=[NX3+]([O-])[a]",
 "nitrogen_mustard":"[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
 "aromatic_hydrazine":"[a][NX3;H1][NX3;H2]","alkyl_sulfonate":"[CX4][OX2][SX4](=[OX1])(=[OX1])"}
PAT = {k: Chem.MolFromSmarts(v) for k, v in TOX.items()}
HETERO_ALERTS = [k for k in TOX if k != "polycyclic_aromatic"]

def eval_set(frame, labels={1}, alerts=None):
    pats = PAT if alerts is None else {k: PAT[k] for k in alerts}
    ev = []
    for smi, lab in zip(frame["canonical"], frame["label"]):
        if lab not in labels: continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None or mol.GetNumAtoms() == 0: continue
        t = set()
        for p in pats.values():
            for mt in mol.GetSubstructMatches(p): t.update(mt)
        if not t or len(t) == mol.GetNumAtoms(): continue
        ev.append({"smiles": smi, "mol": mol, "tox": t, "n_atoms": mol.GetNumAtoms()})
    return ev, np.array([len(e["tox"])/e["n_atoms"] for e in ev])

def degree_of(ev):
    return [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in ev]

def overlap_vec(ev, scores, k_frac, tie_seed):
    rng = np.random.default_rng(tie_seed); out = np.full(len(ev), np.nan)
    for i, (e, sc) in enumerate(zip(ev, scores)):
        if sc is None or len(sc) != e["n_atoms"]: continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[i] = len(set(order[:k].tolist()) & e["tox"])/k
    return out

def mean_overlap(ev, scores, n=N_TIEBREAK, base=0):
    return np.nanmean([overlap_vec(ev, scores, K_FRAC, base+j) for j in range(n)], axis=0)

def lift_of(ov, ch):
    m = ~np.isnan(ov)
    return float(ov[m].mean()/ch[m].mean()) if m.sum() else np.nan

def per_molecule_auroc(ev, scores):
    out = np.full(len(ev), np.nan)
    for i, (e, sc) in enumerate(zip(ev, scores)):
        if sc is None or len(sc) != e["n_atoms"]: continue
        y = np.zeros(e["n_atoms"]); y[list(e["tox"])] = 1
        if 0 < y.sum() < len(y): out[i] = _auc(y, np.asarray(sc, dtype=float))
    return out

def tie_fraction(ev, scores, k_frac=K_FRAC):
    out = []
    for e, sc in zip(ev, scores):
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        v = np.sort(np.asarray(sc, dtype=float))[::-1]; cut = v[k-1]
        above = int((v > cut).sum()); at = int((v == cut).sum())
        out.append(0.0 if at == 0 else (k - above)/k)
    return np.array(out)

def ig_scores(model, graphs, n_steps=N_IG_STEPS, bs=64, signed=False):
    was, saved = model.training, [p.requires_grad for p in model.parameters()]
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    out = []
    for batch in DataLoader(graphs, batch_size=bs, shuffle=False):
        batch = batch.to(DEVICE); x0 = batch.x.clone()
        total = torch.zeros_like(x0)
        for al in torch.linspace(1.0/n_steps, 1.0, n_steps):
            xa = (x0*al).detach().requires_grad_(True)
            lg = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
            total += torch.autograd.grad(lg.sum(), xa)[0]
        a_ = x0*(total/n_steps)
        attr = (a_.sum(1) if signed else a_.abs().sum(1)).detach().cpu().numpy()
        b = batch.batch.cpu().numpy()
        for gi in range(batch.num_graphs): out.append(attr[b == gi])
    for p, f in zip(model.parameters(), saved): p.requires_grad_(f)
    model.train(was)
    return out

def cached_ig(sd, tag, model, graphs, idx, signed=False):
    """Attribution scores are stored, so later sections reuse them instead of recomputing.

    The SMILES are stored with them and checked on load. A cache keyed only by position is
    silently wrong the moment the evaluation set changes, which is the failure the assertion
    in the subsampling reference guards against for the older files."""
    p = f"{EXPL}/p17_{tag}_part{sd}_m{idx}{'_signed' if signed else ''}.npz"
    smiles = np.array([g.smiles for g in graphs])
    if os.path.exists(p):
        z = np.load(p, allow_pickle=True)
        if "smiles" in z.files and len(z["smiles"]) == len(smiles) \
                and (z["smiles"] == smiles).all():
            return [z[f"m{i}"] for i in range(len(graphs))]
        print(f"    cache at {os.path.basename(p)} holds a different molecule set, recomputing")
    sc = ig_scores(model, graphs, signed=signed)
    np.savez_compressed(p, smiles=smiles, **{f"m{i}": s for i, s in enumerate(sc)})
    return sc

def load_partition_models(sd):
    out = []
    for f in sorted(glob.glob(f"{MODELS}/gin_part{sd}_seed*.pt")):
        mdl = build_gnn("gin", HP["hidden"], HP["n_layers"], HP["dropout"],
                        HP.get("residual", True))
        mdl.load_state_dict(torch.load(f, map_location=DEVICE,
                                       weights_only=False)["state_dict"])
        mdl.eval(); out.append(mdl)
    return out

def paired_diff(per, ref, n_boot=N_BOOT, seed=SEED):
    """Molecules and models resampled together, difference taken within each replicate."""
    rng = np.random.default_rng(seed); n_mod, n_mol = per.shape
    out = np.empty(n_boot)
    for b in range(n_boot):
        mi = rng.integers(0, n_mol, n_mol); si = rng.integers(0, n_mod, n_mod)
        out[b] = np.nanmean(per[np.ix_(si, mi)]) - np.nanmean(ref[mi])
    return out

def perfect_ranking(ev):
    return [np.array([1.0 if a in e["tox"] else 0.0 for a in range(e["n_atoms"])])
            for e in ev]

def build_headline():
    """Degree, a random ranking and the trained models, on every partition.

    The random arm is not decoration: the claim that the measure behaves as it should rests
    on degree sitting above it, and without it a shifting degree AUROC cannot be told from a
    shifting measure."""
    rows = []
    for sd in PART_SEEDS:
        te = PARTS[sd][2]
        ev, ch = eval_set(te)
        dg = degree_of(ev)
        graphs = [smiles_to_data(e["smiles"], 1.0) for e in ev]
        d_auc = per_molecule_auroc(ev, dg)
        rng = np.random.default_rng(SEED + sd)
        rnd = [rng.random(e["n_atoms"]) for e in ev]
        r_auc = per_molecule_auroc(ev, rnd)
        ceiling = float(np.nanmean(per_molecule_auroc(ev, perfect_ranking(ev))))
        # a ranking that puts every alert atom first separates perfectly, so this is 1.0
        # by construction; if it is not, the scoring path is broken
        assert abs(ceiling - 1.0) < 1e-9, f"partition {sd}: ceiling AUROC is {ceiling:.6f}"
        mdls = load_partition_models(sd)
        assert len(mdls) == N_MODELS_PER_PARTITION, \
            f"partition {sd}: {len(mdls)} models on disk, expected {N_MODELS_PER_PARTITION}"
        per = np.array([per_molecule_auroc(ev, cached_ig(sd, "ig", m, graphs, k))
                        for k, m in enumerate(mdls)])
        d = paired_diff(per, d_auc, seed=SEED + sd)
        lo, hi = np.percentile(d, [2.5, 97.5])
        rows.append({"partition": sd, "n_eval": len(ev), "n_models": len(mdls),
                     "chance": float(ch.mean()),
                     "random_auroc": float(np.nanmean(r_auc)),
                     "degree_auroc": float(np.nanmean(d_auc)),
                     "gin_auroc": float(np.nanmean(per)),
                     "ceiling_auroc": ceiling,
                     "degree_pct_of_range": 100*(np.nanmean(d_auc)-0.5)/(ceiling-0.5),
                     "gin_pct_of_range": 100*(np.nanmean(per)-0.5)/(ceiling-0.5),
                     "difference": float(np.nanmean(per) - np.nanmean(d_auc)),
                     "ci_low": lo, "ci_high": hi,
                     "excludes_zero": bool(lo > 0 or hi < 0),
                     "degree_lift": lift_of(mean_overlap(ev, dg, base=30000+sd), ch),
                     "degree_tie_fraction": float(np.mean(tie_fraction(ev, dg)))})
        for m in mdls: del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f"  partition {sd}: n={len(ev)}, random {np.nanmean(r_auc):.4f}, "
              f"degree {np.nanmean(d_auc):.4f}, GIN {np.nanmean(per):.4f}, "
              f"diff {rows[-1]['difference']:+.4f} [{lo:+.4f}, {hi:+.4f}], "
              f"tie {rows[-1]['degree_tie_fraction']:.3f}")
    return pd.DataFrame(rows)

head = cached("table87_headline_by_partition.csv", build_headline)
print()
print(head.round(4).to_string(index=False))
print()
print("  Original partition: random 0.492, degree 0.580, GIN 0.637,")
print("  difference +0.056 [+0.019, +0.093], tie fraction 0.868.")


def build_subsample_reference():
    """What the original interval looks like at five models rather than twenty.

    The comparison above uses five models per partition and the original used twenty, so a
    wider interval here would be expected whether or not the effect changed. Subsampling the
    twenty repeatedly gives the reference the comparison needs."""
    fs = [f for f in sorted(glob.glob(f"{EXPL}/scores_ig_seed*.npz"))
          if "_gat_" not in f and "_gcn_" not in f and "no_degree" not in f]
    if len(fs) < 20:
        return pd.DataFrame([{"note": "original attribution scores not found"}])
    prim = pd.read_csv(f"{R}/primary_test.csv")
    ev, ch = eval_set(prim)
    d_auc = per_molecule_auroc(ev, degree_of(ev))
    def load_one(f):
        z = np.load(f)                      # opened once per file, not once per molecule
        return per_molecule_auroc(ev, [z[f"m{i}"] for i in range(len(ev))])
    per_all = np.array([load_one(f) for f in fs])
    # the stored key order must match the evaluation set this cell rebuilds; if it does
    # not, every number below is silently wrong
    assert abs(np.nanmean(per_all) - 0.6368) < 0.005, (
        f"stored scores give {np.nanmean(per_all):.4f}, published GIN AUROC is 0.6368; "
        "the key order of the npz files does not match this evaluation set")
    rng = np.random.default_rng(SEED)
    rows = []
    for n_ in (5, 20):
        widths, diffs = [], []
        for rep in range(40 if n_ < 20 else 1):
            sel = rng.choice(len(per_all), n_, replace=False) if n_ < 20 else np.arange(20)
            d = paired_diff(per_all[sel], d_auc, n_boot=2000, seed=SEED+rep)
            lo, hi = np.percentile(d, [2.5, 97.5])
            widths.append(hi-lo); diffs.append(np.nanmean(per_all[sel])-np.nanmean(d_auc))
        rows.append({"n_models": n_, "n_repeats": len(widths),
                     "mean_difference": float(np.mean(diffs)),
                     "mean_ci_width": float(np.mean(widths)),
                     "sd_of_difference": float(np.std(diffs, ddof=1)) if len(diffs) > 1 else np.nan})
    return pd.DataFrame(rows)

sub = cached("table87b_subsample_reference.csv", build_subsample_reference)
print()
print(sub.round(4).to_string(index=False))
print()
print("  The five-model row is the like-for-like reference for the three partitions above.")

  table87_headline_by_partition.csv: loaded from disk

 partition  n_eval  n_models  chance  random_auroc  degree_auroc  gin_auroc  ceiling_auroc  degree_pct_of_range  gin_pct_of_range  difference  ci_low  ci_high  excludes_zero  degree_lift  degree_tie_fraction
       201     302         5  0.4536        0.4969        0.6070     0.6398            1.0              21.4048           27.9555      0.0328 -0.0024   0.0650          False       1.2154               0.8491
       202     349         5  0.4460        0.4993        0.5712     0.6464            1.0              14.2315           29.2747      0.0752  0.0175   0.1243           True       1.2465               0.8948
       203     257         5  0.3958        0.4955        0.5360     0.6717            1.0               7.2084           34.3397      0.1357  0.0876   0.1910           True       1.1969               0.9052

  Original partition: random 0.492, degree 0.580, GIN 0.637,
  difference +0.056 [+0.019, +0.093], tie fraction 

In [8]:
# What the acyclic exclusion costs the alert-library argument.
#
# Both splitters put the acyclic group in training, so no evaluation set in this work
# contains an acyclic compound. The manuscript's regime argument rests on the fused-aromatic
# pattern supplying 61.8% of alert atoms, a figure measured on a set that by construction
# has no purely aliphatic mutagens in it. The same share over all mutagenic compounds says
# how much of that imbalance is the library and how much is the split.

def alert_atom_shares(frame):
    tot, poly = 0, 0
    n_mol, n_acyc = 0, 0
    for smi, lab in zip(frame.canonical, frame.label):
        if lab != 1: continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None or mol.GetNumAtoms() == 0: continue
        atoms = {k: set() for k in TOX}
        for k, p in PAT.items():
            for mt in mol.GetSubstructMatches(p): atoms[k].update(mt)
        allt = set().union(*atoms.values()) if atoms else set()
        if not allt or len(allt) == mol.GetNumAtoms(): continue
        n_mol += 1
        if safe_scaffold(smi) == "": n_acyc += 1
        tot += len(allt); poly += len(atoms["polycyclic_aromatic"])
    return {"n_molecules": n_mol, "acyclic": n_acyc,
            "polycyclic_share_pct": 100*poly/tot if tot else np.nan}

share_all = alert_atom_shares(df)
share_prim = alert_atom_shares(pd.read_csv(f"{R}/primary_test.csv"))
sh = pd.DataFrame([{"set": "all mutagenic compounds", **share_all},
                   {"set": "designated evaluation set", **share_prim}])
sh.to_csv(f"{R}/table85c_alert_share.csv", index=False)
print(sh.round(2).to_string(index=False))
print()
print("  The manuscript quotes 61.8% from the second row. The difference between the rows")
print("  is what the acyclic exclusion contributes, and it belongs in the limitations.")

                      set  n_molecules  acyclic  polycyclic_share_pct
  all mutagenic compounds         2884      351                 46.46
designated evaluation set          237        0                 61.76

  The manuscript quotes 61.8% from the second row. The difference between the rows
  is what the acyclic exclusion contributes, and it belongs in the limitations.


## 5. The regime decomposition, per partition

The two effects the abstract reports — a trained model far above the baseline on the
heteroatom alerts and below chance on the all-carbon pattern — are the most striking result
in the paper and come from the same single partition. Repeating them is the point of this
notebook.

In [9]:
# A published encoding of the polycyclic aromatic alert would be the right control here.
# We do not have one we can verify from this environment: the pattern we first wrote for
# this cell, c1ccc2cc3ccccc3cc2c1, is anthracene, and the Supporting Information of this
# manuscript already says why a named linear acene is the wrong query -- it misses
# phenanthrene, chrysene and benzo[a]pyrene, which are fused angularly and carry the
# mechanism. Using it as a check would have tested the encoding this work rejects.
# The ring-perception arm below does the job the published pattern would have done.

def fused_aromatic_atoms(mol, min_rings=3):
    """Atoms of a fused aromatic system of at least min_rings rings, by ring perception.

    A named substructure query matches one topology: an anthracene pattern misses
    phenanthrene and benzo[a]pyrene, whose rings are fused angularly, and those carry the
    mechanism. Ring perception captures the class the alert stands for."""
    ri = mol.GetRingInfo()
    arom = [set(r) for r in ri.AtomRings()
            if all(mol.GetAtomWithIdx(a).GetIsAromatic() for a in r)]
    if not arom: return set()
    parent = list(range(len(arom)))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for a in range(len(arom)):
        for b in range(a+1, len(arom)):
            if arom[a] & arom[b]: parent[find(a)] = find(b)
    groups = {}
    for a in range(len(arom)): groups.setdefault(find(a), []).append(a)
    out = set()
    for members in groups.values():
        if len(members) >= min_rings:
            for a in members: out |= arom[a]
    return out

def eval_set_custom(frame, atoms_fn):
    ev = []
    for smi, lab in zip(frame["canonical"], frame["label"]):
        if lab != 1: continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None or mol.GetNumAtoms() == 0: continue
        t = atoms_fn(mol)
        if not t or len(t) == mol.GetNumAtoms(): continue
        ev.append({"smiles": smi, "mol": mol, "tox": t, "n_atoms": mol.GetNumAtoms()})
    return ev, np.array([len(e["tox"])/e["n_atoms"] for e in ev])

def smarts_atoms(pat):
    def f(mol):
        t = set()
        for mt in mol.GetSubstructMatches(pat): t.update(mt)
        return t
    return f

REGIME_DEFS = [
    ("no_polycyclic",   lambda te: eval_set(te, alerts=HETERO_ALERTS)),
    ("polycyclic_only", lambda te: eval_set(te, alerts=["polycyclic_aromatic"])),
    ("fused_2plus",     lambda te: eval_set_custom(te, lambda m: fused_aromatic_atoms(m, 2))),
    ("fused_3plus",     lambda te: eval_set_custom(te, lambda m: fused_aromatic_atoms(m, 3))),
]

def build_regimes():
    """Each regime on each partition, with the tie-break count held equal across arms.

    The degree ranking carries about 87% ties at the cut and the attributions about 19%, so
    the two arms are not equally sensitive to the tie-break seed. Using the same count for
    both keeps that difference out of the interval, at the cost of more computation on the
    arm that needs it less."""
    N_TIE_REG = 200
    rows = []
    for sd in PART_SEEDS:
        te = PARTS[sd][2]
        mdls = load_partition_models(sd)
        assert len(mdls) == N_MODELS_PER_PARTITION, \
            f"partition {sd}: {len(mdls)} models on disk"
        for reg, fn in REGIME_DEFS:
            ev, ch = fn(te)
            if len(ev) < 20:
                rows.append({"partition": sd, "regime": reg, "n_eval": len(ev),
                             "note": "fewer than 20 molecules, not computed"})
                print(f"    {sd} / {reg:16s} only {len(ev)} molecules, recorded and skipped")
                continue
            dg = degree_of(ev)
            graphs = [smiles_to_data(e["smiles"], 1.0) for e in ev]
            d_ov = mean_overlap(ev, dg, n=N_TIE_REG, base=31000+sd)
            d_l = lift_of(d_ov, ch)
            ovs = np.array([mean_overlap(ev, cached_ig(sd, f"ig_{reg}", m, graphs, k_),
                                         n=N_TIE_REG, base=31000+sd)
                            for k_, m in enumerate(mdls)])
            m_l = lift_of(np.nanmean(ovs, axis=0), ch)
            rng = np.random.default_rng(SEED + sd)
            bb = np.empty(N_BOOT)
            for b in range(N_BOOT):
                mi = rng.integers(0, len(ev), len(ev))
                si = rng.integers(0, len(ovs), len(ovs))
                mo = np.nanmean(ovs[np.ix_(si, mi)], axis=0); c = ch[mi]
                m1 = ~np.isnan(mo); m2 = ~np.isnan(d_ov[mi])
                bb[b] = mo[m1].mean()/c[m1].mean() - d_ov[mi][m2].mean()/c[m2].mean()
            lo, hi = np.percentile(bb, [2.5, 97.5])
            # a bootstrap p of exactly zero is not a p value
            n_le = int((bb <= 0).sum()); n_ge = int((bb >= 0).sum())
            p_raw = 2*min((1+n_le)/(1+N_BOOT), (1+n_ge)/(1+N_BOOT))
            rows.append({"partition": sd, "regime": reg, "n_eval": len(ev),
                         "n_tiebreaks": N_TIE_REG,
                         "degree_lift": d_l, "gin_lift": m_l,
                         "difference": m_l - d_l, "ci_low": lo, "ci_high": hi,
                         "p_raw": p_raw, "note": ""})
            print(f"    {sd} / {reg:16s} n={len(ev):3d} diff {m_l-d_l:+.3f} "
                  f"[{lo:+.3f}, {hi:+.3f}]  p={p_raw:.4f}")
        for m in mdls: del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    out = pd.DataFrame(rows)
    # Holm across the family actually tested here: every computed regime-partition cell
    ok = out.p_raw.notna() if "p_raw" in out.columns else pd.Series([], dtype=bool)
    if ok.any():
        p = out.loc[ok, "p_raw"].values
        order = np.argsort(p); k = len(p); run = 0.0; adj = np.empty(k)
        for r_, i_ in enumerate(order):
            run = max(run, (k - r_)*p[i_]); adj[i_] = min(run, 1.0)
        out.loc[ok, "p_holm"] = adj
        out.loc[ok, "resolved"] = adj < 0.05
        print(f"\n  Holm applied across {k} regime-partition tests. The manuscript's own")
        print("  regime table corrects across six, so this family is stricter and a cell")
        print("  resolved there but not here has not necessarily changed.")
    return out

reg = cached("table88_regimes_by_partition.csv", build_regimes)
print()
print(reg.round(3).to_string(index=False))
print()
print("  Original partition, our own encoding: heteroatom +0.442 [+0.304, +0.588],")
print("  all-carbon -0.231 [-0.325, -0.144]. The fused_2plus and fused_3plus rows test")
print("  whether the second effect depends on how the alert was written.")
print()
print("  Two caveats on those two rows. They are not the complement of no_polycyclic,")
print("  which is defined by the SMARTS list, so the regimes here do not partition the")
print("  library. And a fused aromatic system found by ring perception need not be all")
print("  carbon: acridine, carbazole and dibenzofuran qualify, so the all-carbon framing")
print("  of the manuscript does not carry over to them.")

    201 / no_polycyclic    n=269 diff +0.479 [+0.297, +0.669]  p=0.0002
    201 / polycyclic_only  n=148 diff -0.309 [-0.413, -0.204]  p=0.0002
    201 / fused_2plus      n=263 diff -0.293 [-0.402, -0.184]  p=0.0002
    201 / fused_3plus      n=120 diff -0.180 [-0.283, -0.075]  p=0.0010
    202 / no_polycyclic    n=299 diff +0.246 [+0.105, +0.396]  p=0.0010
    202 / polycyclic_only  n=107 diff -0.125 [-0.243, -0.013]  p=0.0300
    202 / fused_2plus      n=171 diff -0.197 [-0.282, -0.128]  p=0.0002
    202 / fused_3plus      n=114 diff -0.146 [-0.229, -0.073]  p=0.0002
    203 / no_polycyclic    n=227 diff +0.622 [+0.352, +0.925]  p=0.0002
    203 / polycyclic_only  n= 92 diff -0.321 [-0.461, -0.186]  p=0.0002
    203 / fused_2plus      n=204 diff -0.425 [-0.528, -0.315]  p=0.0002
    203 / fused_3plus      n=125 diff -0.317 [-0.411, -0.219]  p=0.0002

  Holm applied across 12 regime-partition tests. The manuscript's own
  regime table corrects across six, so this family is stricter an

## 6. Difference of differences for the specificity check

The manuscript states that the reading fixed in advance was for the two differences to
separate, that they do not, and that the difference between them was not tested. The test is
cheap and the data are stored: resample molecules within each set and models across both, and
take the difference of the two differences within each replicate. Reporting a comparison one
has characterised in prose but not measured is not a position this paper can hold.

In [10]:
def build_dod():
    prim = pd.read_csv(f"{R}/primary_test.csv")
    files = [f for f in sorted(glob.glob(f"{EXPL}/scores_ig_seed*.npz"))
             if "_gat_" not in f and "_gcn_" not in f and "no_degree" not in f]
    hp = HP
    rows = []
    found_any = False
    for arch in ["gin", "gat", "gcn"]:
        fs = sorted(glob.glob(f"{MODELS}/{arch}_attr_seed*.pt"))[:10]
        if not fs:
            print(f"    {arch}: no stored weights, skipped"); continue
        found_any = True
        hpa = json.load(open(f"{R}/hpo_gnn.json"))[arch]
        pos_ev, pos_ch = eval_set(prim, {1})
        neg_ev, neg_ch = eval_set(prim, {0})
        pos_g = [smiles_to_data(e["smiles"], 1.0) for e in pos_ev]
        neg_g = [smiles_to_data(e["smiles"], 0.0) for e in neg_ev]
        d_pos = per_molecule_auroc(pos_ev, degree_of(pos_ev))
        d_neg = per_molecule_auroc(neg_ev, degree_of(neg_ev))
        P, N = [], []
        for f in fs:
            mdl = build_gnn(arch, hpa["hidden"], hpa["n_layers"], hpa["dropout"],
                            hpa.get("residual", True))
            mdl.load_state_dict(torch.load(f, map_location=DEVICE,
                                           weights_only=False)["state_dict"]); mdl.eval()
            P.append(per_molecule_auroc(pos_ev, ig_scores(mdl, pos_g, signed=True)))
            N.append(per_molecule_auroc(neg_ev, ig_scores(mdl, neg_g, signed=True)))
            del mdl
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        P, N = np.array(P), np.array(N)
        rng = np.random.default_rng(SEED)
        bb = np.empty(N_BOOT)
        for b in range(N_BOOT):
            si = rng.integers(0, P.shape[0], P.shape[0])       # models shared
            pi = rng.integers(0, P.shape[1], P.shape[1])
            ni = rng.integers(0, N.shape[1], N.shape[1])
            dp = np.nanmean(P[np.ix_(si, pi)]) - np.nanmean(d_pos[pi])
            dn = np.nanmean(N[np.ix_(si, ni)]) - np.nanmean(d_neg[ni])
            bb[b] = dn - dp
        lo, hi = np.percentile(bb, [2.5, 97.5])
        dp0 = float(np.nanmean(P) - np.nanmean(d_pos))
        dn0 = float(np.nanmean(N) - np.nanmean(d_neg))
        rows.append({"architecture": arch.upper(), "n_models": len(fs),
                     "diff_mutagenic": dp0, "diff_non_mutagenic": dn0,
                     "dod": dn0 - dp0,                    # from the data, not the bootstrap
                     "dod_bootstrap_mean": float(np.nanmean(bb)),
                     "ci_low": lo, "ci_high": hi,
                     "excludes_zero": bool(lo > 0 or hi < 0),
                     "reading": ("supported" if hi < 0 else
                                 "opposite direction" if lo > 0 else "not resolved")})
        print(f"    {arch}: dod {np.nanmean(bb):+.4f} [{lo:+.4f}, {hi:+.4f}]")
    assert found_any, "no stored weights for any architecture; check the filenames in MODELS"
    return pd.DataFrame(rows)

dod = cached("table89_difference_of_differences.csv", build_dod)
print()
print(dod.round(4).to_string(index=False))
print()
print("  The statistic is (non-mutagenic - degree) - (mutagenic - degree), so the")
print("  pre-registered direction is negative. The bootstrap mean is reported beside the")
print("  point estimate so that any bootstrap bias is visible rather than hidden.")

  table89_difference_of_differences.csv: loaded from disk

architecture  n_models  diff_mutagenic  diff_non_mutagenic     dod  dod_bootstrap_mean  ci_low  ci_high  excludes_zero      reading
         GIN        10          0.0890              0.0393 -0.0498             -0.0497 -0.0907  -0.0114           True    supported
         GAT        10         -0.0317             -0.0554 -0.0238             -0.0236 -0.0650   0.0184          False not resolved
         GCN        10          0.0310              0.0314  0.0004              0.0003 -0.0427   0.0426          False not resolved

  The statistic is (non-mutagenic - degree) - (mutagenic - degree), so the
  pre-registered direction is negative. The bootstrap mean is reported beside the
  point estimate so that any bootstrap bias is visible rather than hidden.


## 7. A matched arm for the permutation control

Models fitted to permuted labels were trained with regularisation and early stopping switched
off and the budget escalated, so that they would absorb the labels. The models they are
compared with were not. The drop from $1.116$ to $1.036$ is therefore consistent with two
accounts, and the manuscript builds an interpretive paragraph on one of them. Fitting real
labels under the same escalated regime separates them.

In [11]:
def build_matched_permutation():
    """Real labels under the regime the permuted arm was fitted in.

    train_gnn is not usable here. Even with patience set beyond the epoch budget it restores
    the best validation epoch before returning, so it selects early without stopping early,
    and it uses a class-weighted loss. The permuted arm used train_to_fit: no validation, no
    selection, unweighted loss, fixed epochs, lowered learning rate. Matching the regime means
    using the same function with the same settings and changing only the labels."""
    prim_tr = pd.read_csv(f"{R}/primary_train.csv")
    prim_te = pd.read_csv(f"{R}/primary_test.csv")
    ev, ch = eval_set(prim_te, {1})
    graphs = [smiles_to_data(e["smiles"], 1.0) for e in ev]
    tr_set = build_dataset(prim_tr)

    # the settings the permutation control actually used
    perm = pd.read_csv(f"{R}/table46_permutation_repeats.csv")
    if "fitted" in perm.columns: perm = perm[perm.fitted == True]
    g = perm[perm.family == "graph"]
    # The permutation control did not store the budget it settled on, but it escalated
    # through a fixed ladder and recorded which rung it stopped at, so the settings are
    # recoverable rather than guessed.
    LADDER = [(HP["lr"], 300), (HP["lr"]/3.0, 400), (HP["lr"]/10.0, 500)]
    if "attempts" in g.columns and g.attempts.notna().any():
        rung = int(g.attempts.dropna().median())
    else:
        # every graph repeat is recorded as fitted, and the first rung is what a run that
        # fits without escalating used
        rung = 1
        print("    the attempt count was not recorded; assuming the first rung")
    rung = min(max(rung, 1), len(LADDER))
    lr, n_epochs = LADDER[rung-1]
    print(f"  matching the permuted arm at rung {rung} of {len(LADDER)}: "
          f"{n_epochs} epochs, lr {lr:.2e}, unweighted loss, no validation, no selection")
    rows, lifts, train_aucs = [], [], []
    for i in range(8):
        # train_to_fit returns the final training loss, not an AUC; the comparison with
        # the permuted arm is on training ROC-AUC, so it is computed here
        m, _loss = train_to_fit("gin", HP, 52000+i, tr_set, n_epochs=n_epochs,
                                dropout=0.0, weight_decay=0.0, lr=lr)
        y_, p_ = evaluate(m, DataLoader(tr_set, batch_size=256))
        train_aucs.append(float(_auc(y_, p_)))
        sc = ig_scores(m, graphs)
        lifts.append(lift_of(mean_overlap(ev, sc, n=N_TIEBREAK, base=32000+i), ch))
        del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f"    matched arm {i+1}/8: lift {lifts[-1]:.4f}  train AUC {train_aucs[-1]:.4f}")
    lifts = np.array(lifts)
    # percentile bootstrap of a mean is anti-conservative at n = 8; the decision rule
    # depends on this interval, so a t interval is used instead
    se = lifts.std(ddof=1)/np.sqrt(len(lifts))
    lo, hi = stats.t.interval(0.95, len(lifts)-1, loc=lifts.mean(), scale=se)
    rows.append({"arm": "real labels, matched regime", "n": len(lifts),
                 "lift_mean": float(lifts.mean()), "lift_sd": float(lifts.std(ddof=1)),
                 "ci_low": lo, "ci_high": hi,
                 "train_auc": float(np.nanmean(train_aucs))})
    gl = g.lift
    rows.append({"arm": "permuted labels, same regime", "n": len(gl),
                 "lift_mean": float(gl.mean()), "lift_sd": float(gl.std(ddof=1)),
                 "ci_low": np.nan, "ci_high": np.nan,
                 "train_auc": float(g.train_auc_permuted.mean())
                              if "train_auc_permuted" in g.columns else np.nan})
    un = pd.read_csv(f"{R}/table12_untrained_null.csv").set_index("arch")
    rows.append({"arm": "untrained", "n": int(un.loc["gin", "n"]),
                 "lift_mean": float(un.loc["gin", "mean"]),
                 "lift_sd": float(un.loc["gin", "sd"]),
                 "ci_low": np.nan, "ci_high": np.nan, "train_auc": np.nan})
    out = pd.DataFrame(rows)
    # The comparison is between this arm and the permuted arm, both in the same regime.
    # The untrained value is not an anchor here: it belongs to a network that was never
    # fitted at all, and the question is whether the drop follows from the labels or from
    # the way the fitting was done.
    perm_v = float(gl.mean())
    verdict = ("labels" if lo > perm_v else
               "regime" if lo <= perm_v <= hi else
               "below the permuted arm, unexpected")
    out.loc[0, "reading"] = verdict
    print(f"\n  matched arm 95% t interval [{lo:.4f}, {hi:.4f}]; permuted arm {perm_v:.4f}")
    print(f"  reading: {verdict}")
    return out

matched = cached("table90_matched_permutation.csv", build_matched_permutation)
print()
print(matched.round(4).to_string(index=False))
print()
print("  The train AUC column is the check on whether the two arms fitted their targets to")
print("  the same degree. If the matched arm sits far above the permuted one, the regimes")
print("  are still not comparable and that has to be said.")

  matching the permuted arm at rung 2 of 3: 400 epochs, lr 6.01e-04, unweighted loss, no validation, no selection
    matched arm 1/8: lift 1.1204  train AUC 0.9999
    matched arm 2/8: lift 1.0578  train AUC 0.9995
    matched arm 3/8: lift 0.9281  train AUC 0.9997
    matched arm 4/8: lift 1.1616  train AUC 0.9996
    matched arm 5/8: lift 1.0359  train AUC 0.9999
    matched arm 6/8: lift 1.1611  train AUC 0.9995
    matched arm 7/8: lift 1.1611  train AUC 0.9999
    matched arm 8/8: lift 1.0895  train AUC 0.9989

  matched arm 95% t interval [1.0214, 1.1575]; permuted arm 1.0359
  reading: regime
  table90_matched_permutation.csv: written

                         arm   n  lift_mean  lift_sd  ci_low  ci_high  train_auc reading
 real labels, matched regime   8     1.0894   0.0814  1.0214   1.1575     0.9996  regime
permuted labels, same regime  14     1.0359   0.0738     NaN      NaN     0.8737     NaN
                   untrained 500     1.1156   0.0588     NaN      NaN        NaN 

## 8. Summary

In [12]:
print("partition overlap")
print(ov.round(4).to_string(index=False))
if len(head):
    print("\nheadline, per partition")
    print(head[["partition","n_eval","random_auroc","degree_auroc","gin_auroc","difference",
                "ci_low","ci_high","excludes_zero","degree_tie_fraction"]]
          .round(4).to_string(index=False))
    print("\ncomposition of the evaluation sets")
    print(comp[comp.set == "test"][["partition","n","acyclic_frac","mutagenic_frac"]]
          .round(4).to_string(index=False))
if len(reg) and "difference" in reg.columns:
    ok = reg[reg.difference.notna()]
    print("\nregimes, per partition")
    print(ok[["partition","regime","n_eval","difference","ci_low","ci_high",
              "p_holm","resolved"]].round(3).to_string(index=False))
    for rg in ok.regime.unique():
        s_ = ok[ok.regime == rg]
        print(f"  {rg}: signs {list(np.sign(s_.difference).astype(int))}, "
              f"{int(s_.resolved.sum())}/{len(s_)} resolved after Holm")
if len(dod) and "dod" in dod.columns:
    print("\ndifference of differences")
    print(dod[["architecture","diff_mutagenic","diff_non_mutagenic","dod",
               "ci_low","ci_high","reading"]].round(4).to_string(index=False))
print("\nmatched permutation")
print(matched.round(4).to_string(index=False))

num = {"n_partitions": float(len(PART_SEEDS)),
       "n_models_per_partition": float(N_MODELS_PER_PARTITION),
       "n_tiebreaks": float(N_TIEBREAK), "n_bootstrap": float(N_BOOT),
       "acyclic_compounds": float(n_acyclic_total)}
for _, r in ov.iterrows(): num[f"overlap_{r['pair'].replace('-','_')}"] = float(r.jaccard)
for _, r in (head.iterrows() if len(head) else []):
    p = int(r.partition)
    num[f"p{p}_degree_auroc"] = float(r.degree_auroc)
    num[f"p{p}_gin_auroc"] = float(r.gin_auroc)
    num[f"p{p}_diff"] = float(r.difference)
    num[f"p{p}_lo"] = float(r.ci_low); num[f"p{p}_hi"] = float(r.ci_high)
    num[f"p{p}_tiefrac"] = float(r.degree_tie_fraction)
for _, r in (reg[reg.difference.notna()].iterrows() if "difference" in reg.columns else []):
    num[f"p{int(r.partition)}_{r.regime}_diff"] = float(r.difference)
    num[f"p{int(r.partition)}_{r.regime}_lo"] = float(r.ci_low)
    num[f"p{int(r.partition)}_{r.regime}_hi"] = float(r.ci_high)
for _, r in dod.iterrows():
    num[f"dod_{r.architecture}"] = float(r.dod)
    num[f"dod_{r.architecture}_lo"] = float(r.ci_low)
    num[f"dod_{r.architecture}_hi"] = float(r.ci_high)
for _, r in matched.iterrows():
    num[f"matched_{r['arm'].split(',')[0].replace(' ','_')}"] = float(r.lift_mean)
pd.DataFrame(sorted(num.items()), columns=["quantity","value"]).to_csv(
    f"{R}/table91_part17_ledger.csv", index=False)
print(f"\n{len(num)} quantities written for merging into the manuscript ledger")
print("Written to", R)
for f in ["table86_partition_overlap.csv","table87_headline_by_partition.csv",
          "table88_regimes_by_partition.csv","table89_difference_of_differences.csv",
          "table90_matched_permutation.csv","table91_part17_ledger.csv"]:
    print("  -", f, "yes" if os.path.exists(f"{R}/{f}") else "MISSING")
import shutil
from google.colab import files
shutil.make_archive("/content/part17_results", "zip", R)
files.download("/content/part17_results.zip")

partition overlap
       pair  jaccard
    201-202   0.0606
    201-203   0.0583
    202-203   0.0708
201-primary   0.0764
202-primary   0.0716
203-primary   0.0852

headline, per partition
 partition  n_eval  random_auroc  degree_auroc  gin_auroc  difference  ci_low  ci_high  excludes_zero  degree_tie_fraction
       201     302        0.4969        0.6070     0.6398      0.0328 -0.0024   0.0650          False               0.8491
       202     349        0.4993        0.5712     0.6464      0.0752  0.0175   0.1243           True               0.8948
       203     257        0.4955        0.5360     0.6717      0.1357  0.0876   0.1910           True               0.9052

composition of the evaluation sets
 partition   n  acyclic_frac  mutagenic_frac
       201 726           0.0          0.5895
       202 726           0.0          0.6074
       203 726           0.0          0.5165

regimes, per partition
 partition          regime  n_eval  difference  ci_low  ci_high  p_holm resolv

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>